# Test for a single combination of parameters 

In [ ]:
import numpy as np   
from scipy.interpolate import interp1d 
import kerrgeopy as kg
from scipy.integrate import cumulative_trapezoid
#=================================================================== 
# ! IMPORTANT: UPDATE THIS PATH FOR YOUR OWN MACHINE!  
# 
# ➤ Please replace it with the full path to your "WD-IMBH-Relativistic-Tides" folder.
#   Example:
#        sys.path.append("/Users/your_username/your_project_folder/WD-IMBH-Relativistic-Tides")
#
#  Tip: Keep this notebook in the same directory as the "WD-IMBH-Relativistic-Tides" folder for easier path management.
import sys 
sys.path.append("/your_username/your_project_folder/WD-IMBH-Relativistic-Tides") 
# sys.path.append("/Users/yang/My_Project_Files_Yang_Yang/WD-IMBH-Relativistic-Tides")
from function_py.fourier_series_compute import fourier_series_compute_function
from function_py.center_align_fun import center_align
from function_py.center_align1_fun import center_align1
from function_py.compute_coeff_1_fun import compute_coeff_1
from function_py.compute_coeff_2_fun import compute_coeff_2
from function_py.compute_coeff_3_fun import compute_coeff_3
from function_py.compute_coeff_4_fun import compute_coeff_4
from function_py.fft_convolve_fun import fft_convolve
from function_py.compute_coeff_classical_1_fun import compute_coeff_classical_1
from function_py.compute_coeff_classical_2_fun import compute_coeff_classical_2
#=================================================================== 


In [5]:
# Define compactness as sqrt(GM / (c^2 R))
def WD_R_M(M_WD):  # Returns radius in units of solar gravitational radii (r_g,⊙)
    return 6120 * (M_WD / 1.44)**(-1/3) * (1 - M_WD / 1.44)**(0.447)   

# Define compactness parameter eta 
def WD_eta(M_WD):
    return (M_WD / WD_R_M(M_WD))**(1/2)

# Fixed parameters: white dwarf mass and black hole mass
M_WD = 0.6
M_BH = 1e5
R_BH = M_BH  # Expressed in units of solar gravitational radii (r_g,⊙)
R_star = WD_R_M(M_WD)  # Also in units of solar gravitational radii (r_g,⊙)
ratio_R_star_R_BH = R_star / R_BH
ratio_sqrt_v_star_c = WD_eta(M_WD)
r_t = ratio_R_star_R_BH * (M_BH / M_WD)**(1/3)

print(ratio_sqrt_v_star_c)
print(ratio_R_star_R_BH)
print(r_t)

MBH = M_BH
MWD = M_WD
a_orbit = -0.0 # Black hole spin parameter
x_orbit = 1  # Equatorial plane

0.009652703591497589
0.06439517230820639
3.543802902362189


In [6]:
# Boundary of unstable orbits
# Original function
for iiii in range(1):
    def Unstable_eq_function(rp, e, a):
        p = rp * (1 + e)
        F = p**(-3) * (
            p**3
            - 2 * (3 + e**2) * p**2
            + (3 + e**2)**2 * p
            - 4 * a**2 * (1 - e**2)**2
        )
        N = 2 / p * (
            -p**2
            + ((3 + e**2) - a**2) * p
            - a**2 * (1 + 3 * e**2)
        )
        C = (a**2 - p)**2
        Delta_x = N**2 - 4 * F * C
        return Delta_x - (2 * F * p**2 / ((1 + e) * (3 - e)) + N)**2

    def F(rp, e, a):
        return Unstable_eq_function(rp, e, a)

    # Compute partial derivatives
    def dF_drp(rp, e, a, h=1e-6):
        return (F(rp + h, e, a) - F(rp - h, e, a)) / (2 * h)

    def dF_de(rp, e, a, h=1e-6):
        return (F(rp, e + h, a) - F(rp, e - h, a)) / (2 * h)

    # Define the right-hand side of the ODE
    def drp_de(e, rp, a):
        """Compute dr_p / de"""
        rp = rp[0]  # scipy.integrate.solve_ivp passes an array
        df_drp = dF_drp(rp, e, a)
        df_de = dF_de(rp, e, a)

        if abs(df_drp) < 1e-12:
            # Derivative near zero → vertical tangent; integration cannot proceed
            return [0.0]

        return [-abs(df_de / df_drp)]

# Integrate using solve_ivp
from scipy.integrate import solve_ivp

def solve_rp_function(a):
    e0 = 0  # Known eccentricity value (circular orbit)
    # Compute initial rp0
    Z1 = 1 + (1 - a**2)**(1/3) * ((1 + a)**(1/3) + (1 - a)**(1/3))
    Z2 = (3 * a**2 + Z1**2)**(1/2)
    rp0 = 3 + Z2 - np.sign(a) * ((3 - Z1) * (3 + Z1 + 2 * Z2))**(1/2)  # Correct rp for e=0
    rp0 = rp0 * 1

    # Forward integration to e = 0.999, backward to e = 0.001
    sol_forward = solve_ivp(
        fun=lambda e, y: drp_de(e, y, a),
        t_span=(e0, 0.999),
        y0=[rp0],
        method='RK45',
        max_step=0.01,
        rtol=1e-6,
        atol=1e-10
    )

    sol_backward = solve_ivp(
        fun=lambda e, y: drp_de(e, y, a),
        t_span=(e0, 0.001),
        y0=[rp0],
        method='RK45',
        max_step=0.01,
        rtol=1e-6,
        atol=1e-12
    )

    # Combine results
    e_full = np.concatenate([sol_backward.t[::-1], sol_forward.t[1:]])
    rp_full = np.concatenate([sol_backward.y[0][::-1], sol_forward.y[0][1:]])

    # Create interpolation function
    rp_interp = interp1d(e_full, rp_full, kind='cubic', fill_value="extrapolate")

    return rp_interp

In [7]:
# This is a dataset containing multiple parameter points.
# You can use the flattened 1D list `flattened_rp_e_list` for fast batch computations.
# ===================================
# To ensure smooth interpolation, first estimate the maximum and minimum rp of unstable orbits.
# ===================================
# Maximum rp (at e = 0)
rp_max_unstable = solve_rp_function(a_orbit)(0) 
# Minimum rp (at e → 1)
rp_min_unstable = solve_rp_function(a_orbit)(1)
# ===================================

# Pericenter distance grid
rp_orbit_table = np.linspace(0.8, 30, int(34))

# The following content is merely a supplement to ensure the sufficiency of points
for iiii in range(1):
    # Refine near the minimum unstable rp (dense sampling close to plunge)
    dense_region_start = rp_min_unstable
    dense_region_end   = rp_min_unstable + (rp_max_unstable - rp_min_unstable) * 1.05
    dense_points       = 10  # Number of additional points
    # Generate fine grid in dense region
    rp_dense = np.linspace(dense_region_start, dense_region_end, dense_points)
    # Merge coarse and fine grids
    rp_orbit_table = np.concatenate([rp_orbit_table, rp_dense])
    # Sort and remove duplicates (to avoid endpoint duplication)
    rp_orbit_table = np.unique(rp_orbit_table)  # Automatically sorts and deduplicates

    # Extra refinement very close to minimum rp
    dense_region_start = rp_min_unstable
    dense_region_end   = rp_min_unstable + (rp_max_unstable - rp_min_unstable) / 5
    dense_points       = 5
    rp_dense = np.linspace(dense_region_start, dense_region_end, dense_points)
    rp_orbit_table = np.concatenate([rp_orbit_table, rp_dense])
    rp_orbit_table = np.unique(rp_orbit_table)

    # Refinement near the upper end of the rp range
    dense_region_start = max(rp_orbit_table) * 0.8
    dense_region_end   = max(rp_orbit_table)
    dense_points       = 5
    rp_dense = np.linspace(dense_region_start, dense_region_end, dense_points)
    rp_orbit_table = np.concatenate([rp_orbit_table, rp_dense])
    rp_orbit_table = np.unique(rp_orbit_table)

    # Refinement near the maximum unstable rp (ISCO-like region)
    dense_region_start = rp_max_unstable - (rp_max_unstable - rp_min_unstable) / 5
    dense_region_end   = rp_max_unstable 
    dense_points       = 5  
    rp_dense = np.linspace(dense_region_start, dense_region_end, dense_points) 
    rp_orbit_table = np.concatenate([rp_orbit_table, rp_dense]) 
    rp_orbit_table = np.unique(rp_orbit_table) 

# Eccentricity grid
e_orbit_table = np.linspace(0.001, 0.96, int(32))

for iiii in range(1):
    # Refinement at low eccentricities
    dense_region_start = min(e_orbit_table)
    dense_region_end   = 0.2
    dense_points       = 10
    e_dense = np.linspace(dense_region_start, dense_region_end, dense_points)
    e_orbit_table = np.concatenate([e_orbit_table, e_dense])
    e_orbit_table = np.unique(e_orbit_table) 

    # Refinement near maximum eccentricity
    dense_region_start = max(e_orbit_table) - 0.05
    dense_region_end   = max(e_orbit_table)
    dense_points       = 10
    e_dense = np.linspace(dense_region_start, dense_region_end, dense_points)
    e_orbit_table = np.concatenate([e_orbit_table, e_dense])
    e_orbit_table = np.unique(e_orbit_table)

In [8]:
# Create a 2D dictionary of (rp, e) pairs
two_orbit_table = {}
for i in range(len(rp_orbit_table)):
    two_orbit_table[i] = {}
    for j in range(len(e_orbit_table)):
        two_orbit_table[i][j] = (rp_orbit_table[i], e_orbit_table[j])

# Flatten into a 1D list for efficient batch processing
flattened_rp_e_list = []
for i in two_orbit_table:
    for j in two_orbit_table[i]:
        rp_val, e_val = two_orbit_table[i][j]
        p_val = rp_val * (1 + e_val)  # semi-latus rectum
        flattened_rp_e_list.append((
            rp_val, e_val, p_val, a_orbit, x_orbit, MBH, MWD,
            ratio_sqrt_v_star_c, ratio_R_star_R_BH
        ))

# Print one sample entry
print(flattened_rp_e_list[2])

(np.float64(0.8), np.float64(0.03193548387096774), np.float64(0.8255483870967742), -0.0, 1, 100000.0, 0.6, 0.009652703591497589, 0.06439517230820639)


In [9]:
def tidal_heating_function(args):
    """
    Function executed by a worker process to compute the integral value for a single task.
    All required data must be passed as part of `args`.
    """
    # Assume args is a tuple containing all necessary input parameters
    # Unpack the task arguments into input parameters
    rp_orbit, e_orbit, p_orbit, a_orbit, x_orbit, MBH, MWD, ratio_sqrt_v_star_c, ratio_R_star_R_BH = args 
    
    try:

        # Stage 1: Compute orbital quantities needed for tidal  
        #===============================================
        for stage1 in range(1):
            #==========================================================
            # Check whether the orbit is a stable, physically allowed orbit
            #==========================================================
            orbit = kg.StableOrbit(a_orbit, p_orbit, e_orbit, x_orbit, (0, np.pi, 0, 0))
            #==========================================================
            # Basic orbital computations
            #==========================================================
            # Constants of motion
            E_orbit, L_orbit, Q_orbit = orbit.constants_of_motion()
            L_orbit = abs(L_orbit)
            K_orbit = (L_orbit - a_orbit * E_orbit)**2 + Q_orbit   
            # Orbital frequencies
            upsilon_r_orbit = orbit.upsilon_r
            upsilon_theta_orbit = orbit.upsilon_theta
            upsilon_phi_orbit = orbit.upsilon_phi  
            # Mino-time and coordinate-time periods
            period_r_mino_orbit = 2 * np.pi / upsilon_r_orbit
            period_theta_mino_orbit = 2 * np.pi / upsilon_theta_orbit
            period_phi_mino_orbit = 2 * np.pi / upsilon_phi_orbit
            # Coordinate trajectories as functions of Mino time
            t_func, r_func, theta_func, phi_func = orbit.trajectory()  
            # Four-velocities as functions of Mino time
            u_t_func, u_r_func, u_theta_func, u_phi_func = orbit.four_velocity() 
            #===============================================
            # Generate uniformly sampled Mino time grid
            #===============================================
            # Single radial period in Mino time
            mino_time_max = period_r_mino_orbit    # This quantity is defined using proper time
            mino_time = np.linspace(0, mino_time_max, int(2 * (4e2)))  # Use an even number of sample points
            #===============================================
            # Compute t, r, theta, phi, and Sigma as functions of Mino time
            #===============================================
            # Spacetime coordinates along the worldline
            t_mino_time = t_func(mino_time)
            r_mino_time = r_func(mino_time)
            theta_mino_time = theta_func(mino_time)
            phi_mino_time = phi_func(mino_time)
            # Four-velocity components along the worldline
            u_t_mino_time = u_t_func(mino_time)
            u_r_mino_time = u_r_func(mino_time)
            u_theta_mino_time = u_theta_func(mino_time)
            u_phi_mino_time = u_phi_func(mino_time)
            # Compute Sigma = r² + a² cos²θ
            Sigma_mino_time = r_mino_time**2 + a_orbit**2 * np.cos(theta_mino_time)**2
            #===============================================
            # Compute proper time τ as a function of Mino time
            #===============================================
            # Proper time evolution via numerical integration
            tau_of_mino_time = cumulative_trapezoid(Sigma_mino_time, mino_time, initial=0.0) 
            tau_mino_time_func = interp1d(mino_time, tau_of_mino_time, kind='linear', fill_value='extrapolate')
            #===============================================
            # Compute proper-time orbital periods
            #===============================================
            period_r_tau_orbit = tau_mino_time_func(period_r_mino_orbit)
            period_theta_tau_orbit = tau_mino_time_func(period_theta_mino_orbit) 
            period_phi_tau_orbit = tau_mino_time_func(period_phi_mino_orbit)
            #===============================================
            # Additional auxiliary quantities
            # From Masaki Ishii et al. (2005)
            # https://journals.aps.org/prd/abstract/10.1103/PhysRevD.71.044017
            #===============================================
            # A and B functions
            A_mino_time = E_orbit * (r_mino_time**2 + a_orbit**2) - a_orbit * L_orbit
            B_mino_time = L_orbit - a_orbit * E_orbit * np.sin(theta_mino_time)**2
            # J1 and J2 functions
            J1_mino_time = (r_mino_time**4
                            - 6 * a_orbit**2 * r_mino_time**2 * np.cos(theta_mino_time)**2
                            + a_orbit**4 * np.cos(theta_mino_time)**4)
            J2_mino_time = r_mino_time**2 - a_orbit**2 * np.cos(theta_mino_time)**2
            # alpha function
            alpha_mino_time = ((K_orbit - a_orbit**2 * np.cos(theta_mino_time)**2) / (r_mino_time**2 + K_orbit))**0.5
            # Delta function
            Delta_mino_time = r_mino_time**2 - 2 * r_mino_time + a_orbit**2
            #===============================================
            # Compute frame-dragging precession angle Psi(λ)
            #===============================================
            Psi_Integrand_mino_time = K_orbit**0.5 * (
                A_mino_time / (r_mino_time**2 + K_orbit)
                + a_orbit * B_mino_time / (K_orbit - a_orbit**2 * np.cos(theta_mino_time)**2)
            )
            # Numerical integration of Psi
            Psi_mino_time = cumulative_trapezoid(Psi_Integrand_mino_time, mino_time, initial=0.0)
            # Compute sines and cosines of relevant angles
            cosPsi_mino_time = np.cos(Psi_mino_time)
            sinPsi_mino_time = np.sin(Psi_mino_time)
            costheta_mino_time = np.cos(theta_mino_time)
            sintheta_mino_time = np.sin(theta_mino_time)


        # Stage 2
        #===============================================
        for stage2 in range(1):
            #==========================================================
            # Compute the Fourier series of 1/r(λ)
            #==========================================================
            t = mino_time * upsilon_r_orbit
            f = 1 / r_mino_time
            N = len(t)
            args1 = (t, f, N)
            Coeff_1dr_Four = fourier_series_compute_function(args1)
            #==========================================================
            # Compute Fourier series for 1/r³ and 1/r⁵
            #==========================================================
            # For 1/r³: convolve three copies of 1/r
            aa = np.convolve(Coeff_1dr_Four, Coeff_1dr_Four, mode='full') 
            Coeff_1dr3_Four = np.convolve(aa, Coeff_1dr_Four, mode='full') 
            # For 1/r⁵: convolve 1/r³ with two more copies of 1/r
            aa = np.convolve(Coeff_1dr3_Four, Coeff_1dr_Four, mode='full') 
            Coeff_1dr5_Four = np.convolve(aa, Coeff_1dr_Four, mode='full') 
            del aa
            #==========================================================
            # Extract the linear (secular) part of Psi(λ)
            #==========================================================
            integral = cumulative_trapezoid(
                Psi_Integrand_mino_time / upsilon_r_orbit,
                mino_time * upsilon_r_orbit,
                initial=0
            ) 
            Psi_mino_average = integral[-1] / (2 * np.pi)
            #==========================================================
            # Compute Fourier series of the oscillatory part of Psi(λ)
            #==========================================================
            t = mino_time * upsilon_r_orbit
            f = Psi_mino_time - Psi_mino_average * mino_time * upsilon_r_orbit
            N = len(t)
            args1 = (t, f, N)
            Coeff_Psi_rest_Four = fourier_series_compute_function(args1)
            Psi_mino_average = Psi_mino_average * upsilon_r_orbit
            #==========================================================
            # Compute Fourier series of exp(±i(Psi - average*λ))
            #==========================================================
            t = mino_time * upsilon_r_orbit
            f = np.exp(complex(0, 1) * (Psi_mino_time - Psi_mino_average * mino_time))
            N = len(t)
            args1 = (t, f, N)
            Coeff_Psi_exp_positive_rest_Four = fourier_series_compute_function(args1)

            t = mino_time * upsilon_r_orbit
            f = np.exp(-complex(0, 1) * (Psi_mino_time - Psi_mino_average * mino_time))
            N = len(t)
            args1 = (t, f, N)
            Coeff_Psi_exp_negative_rest_Four = fourier_series_compute_function(args1)

            # Thus, we now have a new fundamental frequency: Psi_mino_average
            # Therefore, Cos(Psi) and Sin(Psi) contain two frequencies:
            #   Psi_mino_average and upsilon_r_orbit

            #==========================================================
            # Construct Fourier coefficients for Cos(Psi) and Sin(Psi)
            #==========================================================
            Coeff_Cos_Psi_Four = [[], [], []]
            Coeff_Cos_Psi_Four[1 + 1] = 1/2 * Coeff_Psi_exp_negative_rest_Four
            Coeff_Cos_Psi_Four[0 + 1] = Coeff_Cos_Psi_Four[1 + 1] * 0
            Coeff_Cos_Psi_Four[-1 + 1] = 1/2 * Coeff_Psi_exp_positive_rest_Four
            Coeff_Cos_Psi_Four = np.array(Coeff_Cos_Psi_Four)   

            Coeff_Sin_Psi_Four = [[], [], []]
            Coeff_Sin_Psi_Four[1 + 1] = -1/(2*complex(0,1)) * Coeff_Psi_exp_negative_rest_Four
            Coeff_Sin_Psi_Four[0 + 1] = Coeff_Sin_Psi_Four[1 + 1] * 0
            Coeff_Sin_Psi_Four[-1 + 1] = 1/(2*complex(0,1)) * Coeff_Psi_exp_positive_rest_Four
            Coeff_Sin_Psi_Four = np.array(Coeff_Sin_Psi_Four)

            #==========================================================
            # Compute Fourier series for cos²(Psi) and sin²(Psi)
            #==========================================================
            from scipy.signal import convolve
            Coeff_Cos_2_Psi_Four = convolve(Coeff_Cos_Psi_Four, Coeff_Cos_Psi_Four, mode='full')   
            Coeff_Sin_2_Psi_Four = convolve(Coeff_Sin_Psi_Four, Coeff_Sin_Psi_Four, mode='full')
            Coeff_Cos_2_Psi_Four = np.array(Coeff_Cos_2_Psi_Four) 
            Coeff_Sin_2_Psi_Four = np.array(Coeff_Sin_2_Psi_Four) 

            #==========================================================
            # Explicitly compute e^(-i2Ψ) = cos(2Ψ) - i sin(2Ψ)
            #==========================================================
            Coeff_Cos_2Psi_Four = Coeff_Cos_2_Psi_Four - Coeff_Sin_2_Psi_Four
            Coeff_Sin_2Psi_Four = 2 * convolve(Coeff_Sin_Psi_Four, Coeff_Cos_Psi_Four, mode='full')    
            Coeff_e_m_2Psi_Four = Coeff_Cos_2Psi_Four - complex(0,1) * Coeff_Sin_2Psi_Four 
            Coeff_e_p_2Psi_Four = Coeff_Cos_2Psi_Four + complex(0,1) * Coeff_Sin_2Psi_Four

            #==========================================================
            # Reshape Coeff_1dr3_Four and Coeff_1dr5_Four into 2D arrays
            # to match the unified tensor format used for angular modes
            #==========================================================
            Coeff_1dr3_Four1 = Coeff_1dr3_Four
            Coeff_1dr3_Four = [[]]
            Coeff_1dr3_Four[0] = Coeff_1dr3_Four1
            del Coeff_1dr3_Four1

            Coeff_1dr5_Four1 = Coeff_1dr5_Four
            Coeff_1dr5_Four = [[]]
            Coeff_1dr5_Four[0] = Coeff_1dr5_Four1
            del Coeff_1dr5_Four1

            Coeff_1dr3_Four = np.array(Coeff_1dr3_Four)
            Coeff_1dr5_Four = np.array(Coeff_1dr5_Four)
            Coeff_Cos_Psi_Four = np.array(Coeff_Cos_Psi_Four)
            Coeff_Sin_Psi_Four = np.array(Coeff_Sin_Psi_Four)
            Coeff_Cos_2_Psi_Four = np.array(Coeff_Cos_2_Psi_Four)   # i.e., cos²(Psi)
            Coeff_Sin_2_Psi_Four = np.array(Coeff_Sin_2_Psi_Four)   # i.e., sin²(Psi)
            Coeff_e_m_2Psi_Four = np.array(Coeff_e_m_2Psi_Four)     # i.e., e^(-i2Ψ)
            Coeff_e_p_2Psi_Four = np.array(Coeff_e_p_2Psi_Four)     # i.e., e^(+i2Ψ)

        # Stage 3
        #===============================================
        for stage3 in range(1):
            #==========================================================
            # Phi_22 component: Coeff_Phi22_mino_Four
            #==========================================================
            Coeff_Phi22_mino_Four_1 = convolve(Coeff_1dr3_Four, Coeff_e_m_2Psi_Four, mode='full') 
            Coeff_Phi22_mino_Four_2 = (L_orbit - a_orbit * E_orbit)**2 * convolve(Coeff_1dr5_Four, Coeff_e_m_2Psi_Four, mode='full') 

            # Pad both terms to the same length before adding
            for step_step in range(1, 1 + 1):
                max_num_in_m1 = max(len(Coeff_Phi22_mino_Four_1[0]), len(Coeff_Phi22_mino_Four_2[0]))
                zero_A = np.zeros(max_num_in_m1)
                
                Coeff_Phi22_mino_Four_10 = [[] for _ in range(len(Coeff_Phi22_mino_Four_1))]
                Coeff_Phi22_mino_Four_20 = [[] for _ in range(len(Coeff_Phi22_mino_Four_2))]
                
                for k in range(len(Coeff_Phi22_mino_Four_10)):
                    Coeff_Phi22_mino_Four_10[k] = center_align(zero_A, Coeff_Phi22_mino_Four_1[k])
                for k in range(len(Coeff_Phi22_mino_Four_20)):
                    Coeff_Phi22_mino_Four_20[k] = center_align(zero_A, Coeff_Phi22_mino_Four_2[k])
                    
                Coeff_Phi22_mino_Four_1 = np.array(Coeff_Phi22_mino_Four_10)
                Coeff_Phi22_mino_Four_2 = np.array(Coeff_Phi22_mino_Four_20)
                del Coeff_Phi22_mino_Four_10, Coeff_Phi22_mino_Four_20
                
                Coeff_Phi22_mino_Four = np.array(Coeff_Phi22_mino_Four_1 + Coeff_Phi22_mino_Four_2)

            #==========================================================
            # Phi_20 component: Coeff_Phi20_mino_Four
            #==========================================================
            Coeff_Phi20_mino_Four_1 = Coeff_1dr3_Four 
            Coeff_Phi20_mino_Four_2 = 3 * (L_orbit - a_orbit * E_orbit)**2 * Coeff_1dr5_Four

            # Pad both terms to the same length before adding
            for step_step in range(1, 1 + 1):
                max_num_in_m1 = max(len(Coeff_Phi20_mino_Four_1[0]), len(Coeff_Phi20_mino_Four_2[0]))
                zero_A = np.zeros(max_num_in_m1)
                
                Coeff_Phi20_mino_Four_10 = [[] for _ in range(len(Coeff_Phi20_mino_Four_1))]
                Coeff_Phi20_mino_Four_20 = [[] for _ in range(len(Coeff_Phi20_mino_Four_2))]
                
                for k in range(len(Coeff_Phi20_mino_Four_10)):
                    Coeff_Phi20_mino_Four_10[k] = center_align(zero_A, Coeff_Phi20_mino_Four_1[k])
                for k in range(len(Coeff_Phi20_mino_Four_20)):
                    Coeff_Phi20_mino_Four_20[k] = center_align(zero_A, Coeff_Phi20_mino_Four_2[k])
                    
                Coeff_Phi20_mino_Four_1 = np.array(Coeff_Phi20_mino_Four_10)
                Coeff_Phi20_mino_Four_2 = np.array(Coeff_Phi20_mino_Four_20)
                del Coeff_Phi20_mino_Four_10, Coeff_Phi20_mino_Four_20
                
                Coeff_Phi20_mino_Four = np.array(Coeff_Phi20_mino_Four_1 + Coeff_Phi20_mino_Four_2)

            #===========================================================================
            # At this point, we have:
            #   Coeff_Phi22_mino_Four 
            #   Coeff_Phi20_mino_Four 
            #==========================================================
            # Ensure both arrays have identical dimensions by padding indices

            # Align second index (m₂ dimension)
            for step_step in range(1, 1 + 1):
                max_num_in_m1 = max(len(Coeff_Phi22_mino_Four[0]), len(Coeff_Phi20_mino_Four[0]))
                zero_A = np.zeros(max_num_in_m1)
                
                Coeff_Phi22_mino_Four_0 = [[] for _ in range(len(Coeff_Phi22_mino_Four))]
                Coeff_Phi20_mino_Four_0 = [[] for _ in range(len(Coeff_Phi20_mino_Four))]
                
                for k in range(len(Coeff_Phi22_mino_Four_0)):
                    Coeff_Phi22_mino_Four_0[k] = center_align(zero_A, Coeff_Phi22_mino_Four[k])
                for k in range(len(Coeff_Phi20_mino_Four_0)):
                    Coeff_Phi20_mino_Four_0[k] = center_align(zero_A, Coeff_Phi20_mino_Four[k])
                    
                Coeff_Phi22_mino_Four = np.array(Coeff_Phi22_mino_Four_0)
                Coeff_Phi20_mino_Four = np.array(Coeff_Phi20_mino_Four_0)
                del Coeff_Phi22_mino_Four_0, Coeff_Phi20_mino_Four_0

            # Align first index (m₁ dimension)
            for step_step in range(1, 1 + 1):
                max_num_in_m1 = max(len(Coeff_Phi22_mino_Four), len(Coeff_Phi20_mino_Four))
                zero_A = np.zeros((max_num_in_m1, len(Coeff_Phi22_mino_Four[0])))
                
                Coeff_Phi22_mino_Four_0 = center_align1(zero_A, Coeff_Phi22_mino_Four)
                Coeff_Phi20_mino_Four_0 = center_align1(zero_A, Coeff_Phi20_mino_Four)
                
                Coeff_Phi22_mino_Four = np.array(Coeff_Phi22_mino_Four_0)
                Coeff_Phi20_mino_Four = np.array(Coeff_Phi20_mino_Four_0)
                del Coeff_Phi22_mino_Four_0, Coeff_Phi20_mino_Four_0

            #==========================================================
            # Truncate m₂ modes beyond |m₂| = 30 to reduce computational cost
            #========================================================== 
            num_m_2 = len(Coeff_Phi22_mino_Four[0])
            num_m_1 = len(Coeff_Phi22_mino_Four)
            H = 30  
            center = int((num_m_2 - 1) / 2)

            Coeff_Phi22_mino_Four1 = [[] for _ in range(num_m_1)]
            for m_1 in range(-int((num_m_1 - 1) / 2), int((num_m_1 - 1) / 2) + 1):
                idx = m_1 + int((num_m_1 - 1) / 2)
                Coeff_Phi22_mino_Four1[idx] = Coeff_Phi22_mino_Four[idx][center - H : center + H + 1]
            Coeff_Phi22_mino_Four1 = np.array(Coeff_Phi22_mino_Four1)

            Coeff_Phi20_mino_Four1 = [[] for _ in range(num_m_1)]
            for m_1 in range(-int((num_m_1 - 1) / 2), int((num_m_1 - 1) / 2) + 1):
                idx = m_1 + int((num_m_1 - 1) / 2)
                Coeff_Phi20_mino_Four1[idx] = Coeff_Phi20_mino_Four[idx][center - H : center + H + 1]
            Coeff_Phi20_mino_Four1 = np.array(Coeff_Phi20_mino_Four1)

            Coeff_Phi22_mino_Four = Coeff_Phi22_mino_Four1
            Coeff_Phi20_mino_Four = Coeff_Phi20_mino_Four1
            del Coeff_Phi22_mino_Four1, Coeff_Phi20_mino_Four1, num_m_1, num_m_2

            #==================================================================
            # Only Coeff_Phi22_mino_Four and Coeff_Phi20_mino_Four are needed
            # for subsequent tidal heating calculations.
            #==================================================================

        # Stage 4  
        #=============================================== 
        for stage4 in range(1):
            #==========================================================
            # Compute the average value of Sigma over one Mino-time radial period: Sigma_average
            #========================================================== 
            # First, fit Sigma_mino_time with a cubic spline interpolant
            Sigma_mino_time_func = interp1d(mino_time, Sigma_mino_time, kind='cubic', fill_value='extrapolate')
            mino_time_fine = np.linspace(0, period_r_mino_orbit, int(1e5))
            integral = cumulative_trapezoid(Sigma_mino_time_func(mino_time_fine), mino_time_fine, initial=0) 
            Sigma_average = integral[-1] / period_r_mino_orbit
            tau_of_mino_time1 = tau_of_mino_time

            #==========================================================
            # Expand the oscillatory part of τ(λ) directly in the exponential,
            # using a small-parameter expansion followed by convolution.
            #========================================================== 
            tau_of_mino_time = tau_of_mino_time1        

            # Prepare a uniform grid and compression parameters
            tt = tau_of_mino_time * upsilon_r_orbit / Sigma_average
            mino_time_fourier = np.linspace(
                0, 
                period_r_tau_orbit * upsilon_r_orbit / Sigma_average, 
                int(1e4)
            ) 
            t1 = mino_time_fourier
            del mino_time_fourier
            N = len(t1)
            Compression_Index = 15   # Order of expansion (e.g., up to 15th power)

            # Compute base Fourier coefficients for the expansion
            Coefficient_tau_lambda_Fourier1 = compute_coeff_1(
                mino_time, tau_of_mino_time, Psi_mino_average, Compression_Index, Sigma_average, tt, t1, N
            )
            Coefficient_tau_lambda_Fourier2 = compute_coeff_2(
                mino_time, tau_of_mino_time, Psi_mino_average, Compression_Index, Sigma_average, tt, t1, N
            )
            Coefficient_tau_lambda_Fourier3 = compute_coeff_3(
                mino_time, tau_of_mino_time, upsilon_r_orbit, Compression_Index, Sigma_average, tt, t1, N
            )
            Coefficient_tau_lambda_Fourier4 = compute_coeff_4(
                mino_time, tau_of_mino_time, upsilon_r_orbit, Compression_Index, Sigma_average, tt, t1, N
            )
            # Truncate each coefficient array to keep only the central 1201 modes (±600)
            H = 600  # Keep a large number here; further truncation will happen later during mode multiplication
            center = int((len(Coefficient_tau_lambda_Fourier1) - 1) / 2)
            Coefficient_tau_lambda_Fourier1 = Coefficient_tau_lambda_Fourier1[center - H : center + H + 1]
            Coefficient_tau_lambda_Fourier2 = Coefficient_tau_lambda_Fourier2[center - H : center + H + 1]
            Coefficient_tau_lambda_Fourier3 = Coefficient_tau_lambda_Fourier3[center - H : center + H + 1]
            Coefficient_tau_lambda_Fourier4 = Coefficient_tau_lambda_Fourier4[center - H : center + H + 1] 
            del tt 
            del t1
            del N
            #==========================================================
            # Perform fast convolution to reconstruct higher-order terms
            # via repeated FFT-based convolution (Compression_Index times)
            #========================================================== 
            # Reconstruct full series for each base coefficient up to order `Compression_Index`
            # 1
            Coefficient_tau_lambda_Fourier1_ = fft_convolve(Coefficient_tau_lambda_Fourier1, Coefficient_tau_lambda_Fourier1)
            for i in range(Compression_Index - 2):
                Coefficient_tau_lambda_Fourier1_ = fft_convolve(Coefficient_tau_lambda_Fourier1_, Coefficient_tau_lambda_Fourier1)
            # 2
            Coefficient_tau_lambda_Fourier2_ = fft_convolve(Coefficient_tau_lambda_Fourier2, Coefficient_tau_lambda_Fourier2)
            for i in range(Compression_Index - 2):
                Coefficient_tau_lambda_Fourier2_ = fft_convolve(Coefficient_tau_lambda_Fourier2_, Coefficient_tau_lambda_Fourier2)
            # 3
            Coefficient_tau_lambda_Fourier3_ = fft_convolve(Coefficient_tau_lambda_Fourier3, Coefficient_tau_lambda_Fourier3)
            for i in range(Compression_Index - 2):
                Coefficient_tau_lambda_Fourier3_ = fft_convolve(Coefficient_tau_lambda_Fourier3_, Coefficient_tau_lambda_Fourier3)
            # 4
            Coefficient_tau_lambda_Fourier4_ = fft_convolve(Coefficient_tau_lambda_Fourier4, Coefficient_tau_lambda_Fourier4)
            for i in range(Compression_Index - 2):
                Coefficient_tau_lambda_Fourier4_ = fft_convolve(Coefficient_tau_lambda_Fourier4_, Coefficient_tau_lambda_Fourier4)

            # Truncate the reconstructed high-order coefficients to match the original width
            H = int((len(Coefficient_tau_lambda_Fourier1) - 1) / 2)  # Preserve full width from base term
            center = int((len(Coefficient_tau_lambda_Fourier1_) - 1) / 2)
            Coefficient_tau_lambda_Fourier1_ = Coefficient_tau_lambda_Fourier1_[center - H : center + H + 1]
            Coefficient_tau_lambda_Fourier2_ = Coefficient_tau_lambda_Fourier2_[center - H : center + H + 1]
            Coefficient_tau_lambda_Fourier3_ = Coefficient_tau_lambda_Fourier3_[center - H : center + H + 1]
            Coefficient_tau_lambda_Fourier4_ = Coefficient_tau_lambda_Fourier4_[center - H : center + H + 1] 

            #==========================================================
            # Group the four base expansions into two conjugate pairs
            #========================================================== 
            num_m_1 = len(Coeff_Phi22_mino_Four)    # Number of m₁ modes (from Ψ precession)
            num_m_2 = len(Coeff_Phi22_mino_Four[0]) # Number of m₂ modes (from radial frequency)
            # Build power tables for positive powers of each base coefficient
            # For m₁-related terms (Ψ frequency)
            Coefficient_tau_lambda_Fourier1_power = [[] for _ in range(int((num_m_1 - 1) / 2) + 1)]
            Coefficient_tau_lambda_Fourier1_power[0] = [1]  # Identity element for convolution
            Coefficient_tau_lambda_Fourier1_power[1] = Coefficient_tau_lambda_Fourier1_

            Coefficient_tau_lambda_Fourier2_power = [[] for _ in range(int((num_m_1 - 1) / 2) + 1)]
            Coefficient_tau_lambda_Fourier2_power[0] = [1]
            Coefficient_tau_lambda_Fourier2_power[1] = Coefficient_tau_lambda_Fourier2_

            # For m₂-related terms (radial frequency)
            Coefficient_tau_lambda_Fourier3_power = [[] for _ in range(int((num_m_2 - 1) / 2) + 1)]
            Coefficient_tau_lambda_Fourier3_power[0] = [1]
            Coefficient_tau_lambda_Fourier3_power[1] = Coefficient_tau_lambda_Fourier3_

            Coefficient_tau_lambda_Fourier4_power = [[] for _ in range(int((num_m_2 - 1) / 2) + 1)]
            Coefficient_tau_lambda_Fourier4_power[0] = [1]
            Coefficient_tau_lambda_Fourier4_power[1] = Coefficient_tau_lambda_Fourier4_
            #==========================================================
            # Recursively compute higher powers via convolution
            #==========================================================
            for i in range(int((num_m_1 - 1) / 2) + 1):
                if i > 1:
                    Coefficient_tau_lambda_Fourier1_power[i] = fft_convolve(
                        Coefficient_tau_lambda_Fourier1_power[i - 1], Coefficient_tau_lambda_Fourier1_
                    )
                    Coefficient_tau_lambda_Fourier2_power[i] = fft_convolve(
                        Coefficient_tau_lambda_Fourier2_power[i - 1], Coefficient_tau_lambda_Fourier2_
                    )

            for i in range(int((num_m_2 - 1) / 2) + 1):
                if i > 1:
                    Coefficient_tau_lambda_Fourier3_power[i] = fft_convolve(
                        Coefficient_tau_lambda_Fourier3_power[i - 1], Coefficient_tau_lambda_Fourier3_
                    )
                    Coefficient_tau_lambda_Fourier4_power[i] = fft_convolve(
                        Coefficient_tau_lambda_Fourier4_power[i - 1], Coefficient_tau_lambda_Fourier4_
                    )
            #==========================================================
            # Truncate all higher powers to the same width as the first-order term
            #==========================================================
            for i in range(int((num_m_1 - 1) / 2) + 1):
                if i > 1:
                    H = int((len(Coefficient_tau_lambda_Fourier1_power[1]) - 1) / 2)
                    center = int((len(Coefficient_tau_lambda_Fourier1_power[i]) - 1) / 2)
                    Coefficient_tau_lambda_Fourier1_power[i] = Coefficient_tau_lambda_Fourier1_power[i][center - H : center + H + 1]
                    Coefficient_tau_lambda_Fourier2_power[i] = Coefficient_tau_lambda_Fourier2_power[i][center - H : center + H + 1]

            for i in range(int((num_m_2 - 1) / 2) + 1):
                if i > 1:
                    H = int((len(Coefficient_tau_lambda_Fourier3_power[1]) - 1) / 2)
                    center = int((len(Coefficient_tau_lambda_Fourier3_power[i]) - 1) / 2)
                    Coefficient_tau_lambda_Fourier3_power[i] = Coefficient_tau_lambda_Fourier3_power[i][center - H : center + H + 1]
                    Coefficient_tau_lambda_Fourier4_power[i] = Coefficient_tau_lambda_Fourier4_power[i][center - H : center + H + 1]
            #==========================================================
            # Combine positive and negative powers into unified dictionaries indexed by (m₁, m₂)
            #==========================================================
            Coefficient_tau_lambda_Fourier_m_1_power = {}
            Coefficient_tau_lambda_Fourier_m_2_power = {} 
            num_m_1_d2 = int((num_m_1 - 1) / 2)
            num_m_2_d2 = int((num_m_2 - 1) / 2)

            for m_1 in range(-num_m_1_d2, num_m_1_d2+1):
                if m_1 >= 0:
                    Coefficient_tau_lambda_Fourier_m_1_power[m_1] = Coefficient_tau_lambda_Fourier1_power[m_1]
                elif m_1 < 0:
                    Coefficient_tau_lambda_Fourier_m_1_power[m_1] = Coefficient_tau_lambda_Fourier2_power[-m_1]   
            for m_2 in range(-num_m_2_d2, num_m_2_d2+1):
                if m_2 >= 0:
                    Coefficient_tau_lambda_Fourier_m_2_power[m_2] = Coefficient_tau_lambda_Fourier3_power[m_2]
                elif m_2 < 0:       
                    Coefficient_tau_lambda_Fourier_m_2_power[m_2] = Coefficient_tau_lambda_Fourier4_power[-m_2]
            
            Coefficient_tau_lambda_Fourier_m_1_power[0] = np.zeros(len(Coefficient_tau_lambda_Fourier_m_1_power[1])) * complex(0, 1) 
            Coefficient_tau_lambda_Fourier_m_1_power[0][int((len(Coefficient_tau_lambda_Fourier_m_1_power[1]) - 1) / 2)] = 1.0 + 0.0 * complex(0, 1)

            Coefficient_tau_lambda_Fourier_m_2_power[0] = np.zeros(len(Coefficient_tau_lambda_Fourier_m_2_power[1])) * complex(0, 1) 
            Coefficient_tau_lambda_Fourier_m_2_power[0][int((len(Coefficient_tau_lambda_Fourier_m_2_power[1]) - 1) / 2)] = 1.0 + 0.0 * complex(0, 1)

        # Stage 5
        #===============================================
        for stage5 in range(1):
            #==========================================================
            # Combine all components to obtain the final decomposition in proper time τ
            #========================================================== 
            num_m_1 = len(Coeff_Phi22_mino_Four)
            num_m_2 = len(Coeff_Phi22_mino_Four[0])
            num_m_1_d2 = int((num_m_1 - 1) / 2)
            num_m_2_d2 = int((num_m_2 - 1) / 2)

            # At this point, we have:
            # - Coefficient_tau_lambda_Fourier_m_1_power[m₁]: Fourier series for exp(i m₁ Ψ_avg τ / Σ_avg)
            # - Coefficient_tau_lambda_Fourier_m_2_power[m₂]: Fourier series for exp(i m₂ ω_r τ / Σ_avg)
            # - Coeff_Phi22_mino_Four[m₁_index][m₂_index]: Mino-time Fourier coefficients of Φ₂₂
            # - Coeff_Phi20_mino_Four[m₁_index][m₂_index]: Mino-time Fourier coefficients of Φ₂₀
            # The angular dependence (Ψ) remains unchanged; only the radial/time reparameterization is updated.

            M_index = int(len(Coefficient_tau_lambda_Fourier_m_1_power[1]))

            # Step 1: Precompute convolution of m₁ and m₂ basis functions for every (m₁, m₂)
            Coefficient_table_A = {}
            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                Coefficient_table_A[m_1] = {}   
                for m_2 in range(-num_m_2_d2, num_m_2_d2 + 1):  
                    A = fft_convolve(
                        Coefficient_tau_lambda_Fourier_m_1_power[m_1],  
                        Coefficient_tau_lambda_Fourier_m_2_power[m_2]
                    )
                    # Truncate result to half its length to control growth
                    center = int((len(A) - 1) / 2)
                    H = int(center / 2)        
                    A = A[center - H : center + H + 1]
                    Coefficient_table_A[m_1][m_2] = A 

            # Step 2: Multiply by Φ₂₂ coefficients and shift in frequency space according to m₂
            Coefficient_table = {}
            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                Coefficient_table[m_1] = np.zeros(M_index + 2 * num_m_2_d2) * complex(0, 1)
                for m_2 in range(-num_m_2_d2, num_m_2_d2 + 1):
                    A = Coefficient_table_A[m_1][m_2] 
                    A = A * Coeff_Phi22_mino_Four[m_1 + num_m_1_d2][m_2 + num_m_2_d2]
                    # Shift spectrum by m₂ bins (to align with global frequency grid)
                    A = np.pad(A, pad_width=(num_m_2_d2 + m_2, num_m_2_d2 - m_2), mode='constant', constant_values=0)
                    Coefficient_table[m_1] += A

            # Step 3: Same for Φ₂₀
            Coefficient_table_1 = {}
            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                Coefficient_table_1[m_1] = np.zeros(M_index + 2 * num_m_2_d2) * complex(0, 1)
                for m_2 in range(-num_m_2_d2, num_m_2_d2 + 1):
                    A = Coefficient_table_A[m_1][m_2] 
                    A = A * Coeff_Phi20_mino_Four[m_1 + num_m_1_d2][m_2 + num_m_2_d2]
                    A = np.pad(A, pad_width=(num_m_2_d2 + m_2, num_m_2_d2 - m_2), mode='constant', constant_values=0)
                    Coefficient_table_1[m_1] += A

            del A

            #==========================================================
            # Final assembly: build global frequency and coefficient arrays
            #========================================================== 
            num_m_1 = len(Coefficient_table)
            num_m_2 = len(Coefficient_table[0])
            num_m_1_d2 = int((num_m_1 - 1) / 2)
            num_m_2_d2 = int((num_m_2 - 1) / 2)

            # Fundamental frequencies in proper time τ
            omega1 = Psi_mino_average / Sigma_average      # From frame-dragging precession
            omega2 = upsilon_r_orbit / Sigma_average       # From radial motion

            # Construct global frequency grid: ω = m₁·ω₁ + m₂·ω₂
            pp = np.arange(-num_m_2_d2, num_m_2_d2 + 1)
            fre_table1 = {}
            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                fre_table1[m_1] = m_1 * omega1 + pp * omega2

            fre_table = []
            for i in range(len(fre_table1)):
                fre_table.append(fre_table1[i - num_m_1_d2])
            del fre_table1  
            fre_table = np.array(fre_table).flatten()

            # Flatten coefficient tables into 1D arrays aligned with fre_table
            Coefficient_Force_scalar_table_1 = []
            Coefficient_Force_scalar_table_2 = []
            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                Coefficient_Force_scalar_table_1.append(Coefficient_table[m_1])
            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                Coefficient_Force_scalar_table_2.append(Coefficient_table_1[m_1])

            Coefficient_Force_scalar_table_1 = np.array(Coefficient_Force_scalar_table_1).flatten()
            Coefficient_Force_scalar_table_2 = np.array(Coefficient_Force_scalar_table_2).flatten()

        # Stage 6: We missed multiplying by a long coefficient in this stage
        #===============================================
        for stage6 in range(1):
            #==========================================================
            # Insert white dwarf physical parameters
            #========================================================== 
            k_rt = rp_orbit / (ratio_R_star_R_BH * (MBH / MWD)**(1/3)) 
            
            # Convert frequencies to stellar-dimensional units
            fre_table_star = fre_table / ratio_sqrt_v_star_c * ratio_R_star_R_BH
            
            # Must multiply by (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
            # Angular momentum flux: proportional to frequency^5 and |coefficient|^2
            dotJ_z_dim = 0
            mm = 2
            W22_square = 3/10 * np.pi 
            dotJ_z_dim += mm * (
                (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
            ) * W22_square * abs(fre_table_star)**5 * abs(Coefficient_Force_scalar_table_1)**2

            # Energy flux: proportional to frequency^6 and |coefficient|^2
            dotE_orb_dim = 0
            # Contribution from l=2, m=2 mode
            dotE_orb_dim += (1/2 * abs(fre_table_star)) * mm * (
                (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
            ) * W22_square * abs(fre_table_star)**5 * abs(Coefficient_Force_scalar_table_1)**2
            
            # Contribution from l=2, m=0 mode
            W20_square = 1/5 * np.pi 
            dotE_orb_dim += (1/2 * abs(fre_table_star)) * mm * (
                (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
            ) * W20_square * abs(fre_table_star)**5 * abs(Coefficient_Force_scalar_table_2)**2

            # ======================================================================
            # Convert results to the form required for Kerr geodesic-based fluxes
            # ========================================================================
            dotJ_z = ratio_sqrt_v_star_c**2 * k_rt**(-6) * dotJ_z_dim
            dotJ_z_sum = sum(dotJ_z)
            
            dotE_orb = ratio_sqrt_v_star_c**5 * k_rt**(-6) * (MBH / MWD) * dotE_orb_dim
            dotE_orb_sum = sum(dotE_orb)
            
            # The above dotJ_z_sum and dotE_orb_sum are computed in the proper time (τ) frame.
            # To convert to coordinate time (t), multiply by <dτ/dt> = period_r_tau_orbit / period_r_t_coord_orbit
            period_r_t_coord_orbit = 2 * np.pi / orbit.omega_r
            dotJ_z_sum = dotJ_z_sum * (period_r_tau_orbit / period_r_t_coord_orbit)
            dotE_orb_sum = dotE_orb_sum * (period_r_tau_orbit / period_r_t_coord_orbit)

        # Stage 7: Compute the corresponding Keplerian orbit (if it exists)
        #===============================================
        for stage7 in range(1):
            #==========================================================
            # Construct a Keplerian orbit with the same pericenter distance `rp_orbit`
            # and the same radial period in coordinate time `period_r_t_coord_orbit`
            #========================================================== 
            # Note: The following two approaches are attempted to define the classical orbit.
            # First approach: match pericenter and eccentricity from the relativistic orbit.
            # Second approach: enforce consistency between period and semi-major axis via Kepler's third law.

            # Use the same eccentricity as the relativistic orbit (thus same apocenter)
            e_classical_orbit = e_orbit
            p_classical_orbit = rp_orbit * (1 + e_classical_orbit)
            # Recompute the orbital period using Kepler's third law: T = 2π a^(3/2), with a = rp / (1 - e)
            period_r_t_coord_orbit = 2 * np.pi * (rp_orbit / (1 - e_classical_orbit))**(3/2)

            # Alternatively, recompute eccentricity from the given period (ensuring Keplerian consistency)
            e_classical_orbit = 1 - rp_orbit / ( (period_r_t_coord_orbit / (2 * np.pi))**(2/3) )
            p_classical_orbit = rp_orbit * (1 + e_classical_orbit)

            #==========================================================
            # Integrate dt/dφ to obtain t(φ) for the Keplerian orbit
            #========================================================== 
            nn_num = 1e3
            # First half-orbit: φ ∈ [0, π]
            phi_net1 = np.linspace(0, np.pi, int(2 * nn_num))
            t_phi_classical_net1 = (p_classical_orbit**3)**0.5 * (
                (2 * np.arctan( ((1 + e_classical_orbit) * np.tan(phi_net1 / 2)) / np.sqrt(1 - e_classical_orbit**2) )) / (1 - e_classical_orbit**2)**1.5
                + (e_classical_orbit * np.sin(phi_net1)) / ((1 - e_classical_orbit**2) * (1 - e_classical_orbit * np.cos(phi_net1)))
            )

            # Second half-orbit: φ ∈ (π, 2π] — avoid singularity at φ = π by offsetting start point
            phi_net2 = np.linspace(np.pi * (1 + 1 / int(2 * nn_num)), 2 * np.pi, int(2 * nn_num))
            t_phi_classical_net2 = (p_classical_orbit**3)**0.5 * (
                (2 * np.arctan( ((1 + e_classical_orbit) * np.tan(phi_net2 / 2)) / np.sqrt(1 - e_classical_orbit**2) )) / (1 - e_classical_orbit**2)**1.5
                + (e_classical_orbit * np.sin(phi_net2)) / ((1 - e_classical_orbit**2) * (1 - e_classical_orbit * np.cos(phi_net2)))
            )
            # Shift second half by one full period to ensure continuity in time
            t_phi_classical_net2 = t_phi_classical_net2 + period_r_t_coord_orbit

            # Assemble full orbital trajectory over φ ∈ [0, 2π]
            phi_classical_orbit = np.concatenate([phi_net1, phi_net2])
            t_classical_orbit   = np.concatenate([t_phi_classical_net1, t_phi_classical_net2])
            # Radial coordinate: r(φ) = p / (1 + e cos(φ + π)) = p / (1 - e cos φ)  [since pericenter at φ=0]
            r_classical_orbit   = p_classical_orbit / (1 + e_classical_orbit * np.cos(phi_classical_orbit + np.pi))

            # Build cubic spline interpolants for r(t) and φ(t)
            r_t_classical_func   = interp1d(t_classical_orbit, r_classical_orbit,   kind='cubic', fill_value='extrapolate')
            phi_t_classical_func = interp1d(t_classical_orbit, phi_classical_orbit, kind='cubic', fill_value='extrapolate')

            # Define the azimuthal angle Ψ (here identical to φ in the Keplerian case)
            Psi_classical_orbit = phi_classical_orbit

            # Compute sine and cosine of Ψ for later use (e.g., waveform projection)
            cosPsi_classical_orbit = np.cos(Psi_classical_orbit)
            sinPsi_classical_orbit = np.sin(Psi_classical_orbit)

            # Clean up temporary arrays
            del phi_net1
            del phi_net2
            del t_phi_classical_net1
            del t_phi_classical_net2


        # Stage 8: Compute Fourier expansions of classical orbital functions
        #===============================================
        for stage8 in range(1):
            #==========================================================
            # First, compute the time-domain data for 1/r_classical_orbit,
            # using phi_classical_orbit as the independent variable (treated as "time" here)
            #========================================================== 
            t = phi_classical_orbit  
            f = 1 / r_classical_orbit
            N = len(t)
            args1 = (t, f, N)
            Coeff_1dr_classical_Four = fourier_series_compute_function(args1)

            #==========================================================
            # Compute Fourier coefficients for (1/r)^3 via convolution:
            # Since multiplication in time domain ↔ convolution in frequency domain,
            # (1/r)^3 corresponds to triple self-convolution of Coeff_1dr_classical_Four.
            #==========================================================
            aa = np.convolve(Coeff_1dr_classical_Four, Coeff_1dr_classical_Four, mode='full') 
            Coeff_1dr3_classical_Four = np.convolve(aa, Coeff_1dr_classical_Four, mode='full') 

            #==========================================================
            # Fourier coefficients for cos(Psi) and sin(Psi)
            # Recall: cos(φ) = (e^{iφ} + e^{-iφ})/2 → [1/2, 0, 1/2] in mode space [-1, 0, 1]
            #          sin(φ) = (e^{iφ} - e^{-iφ})/(2i) → [-1/(2i), 0, 1/(2i)] → represented as real array scaled by 1/(2i)
            #==========================================================
            Coeff_Cos_Psi_classical_Four = np.array([1/2, 0, 1/2]) + 0j  # explicitly complex
            Coeff_Sin_Psi_classical_Four = np.array([-1, 0, 1]) / (2j)

            #==========================================================
            # Fourier coefficients for cos(2Ψ) and sin(2Ψ) via convolution:
            # cos(2Ψ) = cos²Ψ - sin²Ψ, but easier via: Re[e^{i2Ψ}] = Re[(e^{iΨ})^2]
            # Here we use: cos(2Ψ) ↔ conv(cosΨ, cosΨ) - conv(sinΨ, sinΨ) (not used directly),
            # but for consistency, we compute via product identities using convolution.
            #==========================================================
            Coeff_Cos_2_Psi_classical_Four = np.convolve(Coeff_Cos_Psi_classical_Four, Coeff_Cos_Psi_classical_Four, mode='full') 
            Coeff_Sin_2_Psi_classical_Four = np.convolve(Coeff_Sin_Psi_classical_Four, Coeff_Sin_Psi_classical_Four, mode='full') 

            #==========================================================
            # Explicitly construct e^{-i2Ψ} = cos(2Ψ) - i sin(2Ψ)
            # Note: cos(2Ψ) = cos²Ψ - sin²Ψ, sin(2Ψ) = 2 sinΨ cosΨ
            # So we recompute using correct trigonometric identities:
            #==========================================================
            # cos(2Ψ) = cos²Ψ - sin²Ψ
            Coeff_Cos_2Psi_classical_Four = (
                np.convolve(Coeff_Cos_Psi_classical_Four, Coeff_Cos_Psi_classical_Four, mode='full')
                - np.convolve(Coeff_Sin_Psi_classical_Four, Coeff_Sin_Psi_classical_Four, mode='full')
            )
            # sin(2Ψ) = 2 sinΨ cosΨ
            Coeff_Sin_2Psi_classical_Four = 2 * np.convolve(Coeff_Sin_Psi_classical_Four, Coeff_Cos_Psi_classical_Four, mode='full')

            # Assemble complex exponentials
            Coeff_e_m_2Psi_classical_Four = Coeff_Cos_2Psi_classical_Four - 1j * Coeff_Sin_2Psi_classical_Four  # e^{-i2Ψ}
            Coeff_e_p_2Psi_classical_Four = Coeff_Cos_2Psi_classical_Four + 1j * Coeff_Sin_2Psi_classical_Four  # e^{+i2Ψ}

            #==========================================================
            # Ensure all coefficient arrays are explicitly NumPy arrays (for type safety and compatibility)
            #==========================================================
            Coeff_1dr_classical_Four = np.array(Coeff_1dr_classical_Four)
            Coeff_1dr3_classical_Four = np.array(Coeff_1dr3_classical_Four)
            Coeff_Cos_Psi_classical_Four = np.array(Coeff_Cos_Psi_classical_Four)
            Coeff_Sin_Psi_classical_Four = np.array(Coeff_Sin_Psi_classical_Four)
            Coeff_Cos_2_Psi_classical_Four = np.array(Coeff_Cos_2_Psi_classical_Four)
            Coeff_Sin_2_Psi_classical_Four = np.array(Coeff_Sin_2_Psi_classical_Four) 
            Coeff_e_m_2Psi_classical_Four = np.array(Coeff_e_m_2Psi_classical_Four)
            Coeff_e_p_2Psi_classical_Four = np.array(Coeff_e_p_2Psi_classical_Four)

        
        # Stage 9: Compute components of the potential Phi
        #===============================================
        for stage9 in range(1):
            #==========================================================
            # Phi_22 component: Fourier coefficients from (1/r)^3 * e^{-i2Ψ}
            #==========================================================
            Coeff_Phi22_mino_classical_Four_1 = np.convolve(
                Coeff_1dr3_classical_Four, 
                Coeff_e_m_2Psi_classical_Four, 
                mode='full'
            )
            max_num_in_m1 = len(Coeff_Phi22_mino_classical_Four_1)
            zero_A = np.zeros(max_num_in_m1)
            Coeff_Phi22_mino_classical_Four_10 = center_align(zero_A, Coeff_Phi22_mino_classical_Four_1)
            Coeff_Phi22_mino_classical_Four_1 = np.array(Coeff_Phi22_mino_classical_Four_10)
            del Coeff_Phi22_mino_classical_Four_10
            Coeff_Phi22_mino_classical_Four = np.array(Coeff_Phi22_mino_classical_Four_1)
            del Coeff_Phi22_mino_classical_Four_1

            #==========================================================
            # Phi_20 component: Fourier coefficients from (1/r)^3 (axisymmetric part, m=0)
            #==========================================================
            Coeff_Phi20_mino_classical_Four_1 = Coeff_1dr3_classical_Four 
            max_num_in_m1 = len(Coeff_Phi20_mino_classical_Four_1)
            zero_A = np.zeros(max_num_in_m1)
            Coeff_Phi20_mino_classical_Four_10 = center_align(zero_A, Coeff_Phi20_mino_classical_Four_1)
            Coeff_Phi20_mino_classical_Four_1 = np.array(Coeff_Phi20_mino_classical_Four_10)
            del Coeff_Phi20_mino_classical_Four_10
            Coeff_Phi20_mino_classical_Four = np.array(Coeff_Phi20_mino_classical_Four_1)
            del Coeff_Phi20_mino_classical_Four_1

            #===========================================================================
            # Summary: We now have two sets of Fourier coefficients:
            #   - Coeff_Phi22_mino_classical_Four  (m ≠ 0, quadrupolar, rotating)
            #   - Coeff_Phi20_mino_classical_Four  (m = 0, axisymmetric)
            #===========================================================================
            # Align their frequency-domain indices to the same length for consistent indexing
            #===========================================================================
            # Note: The loop `for Complete_the_first_index in range(1,1+1)` runs only once;
            # it is kept for structural consistency but could be removed.
            for Complete_the_first_index in range(1, 1 + 1):
                max_num_in_m1 = max(
                    len(Coeff_Phi22_mino_classical_Four), 
                    len(Coeff_Phi20_mino_classical_Four)
                )
                zero_A = np.zeros(max_num_in_m1)
                
                Coeff_Phi22_mino_classical_Four_0 = center_align(zero_A, Coeff_Phi22_mino_classical_Four)
                Coeff_Phi20_mino_classical_Four_0 = center_align(zero_A, Coeff_Phi20_mino_classical_Four)
                
                Coeff_Phi22_mino_classical_Four = np.array(Coeff_Phi22_mino_classical_Four_0)
                Coeff_Phi20_mino_classical_Four = np.array(Coeff_Phi20_mino_classical_Four_0)
                del Coeff_Phi22_mino_classical_Four_0
                del Coeff_Phi20_mino_classical_Four_0

            #==========================================================
            # Truncate both coefficient arrays to retain only modes with |m₁| ≤ 30
            # (i.e., keep 61 central modes: from -30 to +30)
            #==========================================================
            num_m_1 = len(Coeff_Phi22_mino_classical_Four)
            H = 30  
            center = int((num_m_1 - 1) / 2)

            # Extract central 2*H+1 modes for Phi_22
            Coeff_Phi22_mino_classical_Four1 = Coeff_Phi22_mino_classical_Four[center - H : center + H + 1]

            # Extract central 2*H+1 modes for Phi_20
            Coeff_Phi20_mino_classical_Four1 = Coeff_Phi20_mino_classical_Four[center - H : center + H + 1]

            # Update references
            Coeff_Phi22_mino_classical_Four = np.array(Coeff_Phi22_mino_classical_Four1)
            Coeff_Phi20_mino_classical_Four = np.array(Coeff_Phi20_mino_classical_Four1)

            # Clean up temporary variables
            del Coeff_Phi22_mino_classical_Four1
            del Coeff_Phi20_mino_classical_Four1
            del num_m_1
            # Note: `num_m_2` is referenced but not defined in this block; ensure it exists upstream or remove if unused.


        # Stage 10: Compute Fourier expansions for the time-reparameterization factor τ(λ)
        # Compression_Index and H are critical parameters controlling expansion order and mode truncation.
        #===============================================
        for stage10 in range(1):
            #==========================================================
            # Compute the average value of Sigma over one orbital period in Mino-like time: 
            #========================================================== 
            # First, construct a cubic spline interpolant for Σ 
            Sigma_time_classical_func = interp1d(
                phi_classical_orbit, 
                1 / ( (p_classical_orbit**-3)**0.5 * (1 + e_classical_orbit * np.cos(phi_classical_orbit + np.pi))**2 ),
                kind='cubic',
                fill_value='extrapolate'
            )
            classical_time_fine = np.linspace(0, 2 * np.pi, int(1e5))
            integral = cumulative_trapezoid(Sigma_time_classical_func(classical_time_fine), classical_time_fine, initial=0) 
            Sigma_classical_average = integral[-1] / (2 * np.pi)

            #==========================================================
            # Expand the oscillatory part of τ(λ) directly in the exponential:
            # Use a small-parameter expansion followed by repeated convolution (via FFT).
            #========================================================== 
            # Prepare a uniform time grid and scaling parameters
            tt = t_classical_orbit / Sigma_classical_average
            mino_time_fourier = np.linspace(
                0, 
                period_r_t_coord_orbit / Sigma_classical_average, 
                int(1e4)
            ) 
            t1 = mino_time_fourier
            del mino_time_fourier
            N = len(t1)
            Compression_Index = 15  # Expansion order (e.g., up to 15th power in the exponential)

            # Compute base Fourier coefficient arrays using helper functions
            Coefficient_tau_lambda_classical_Fourier1 = compute_coeff_classical_1(
                phi_classical_orbit, t_classical_orbit, Compression_Index, Sigma_classical_average, tt, t1, N
            )
            Coefficient_tau_lambda_classical_Fourier2 = compute_coeff_classical_2(
                phi_classical_orbit, t_classical_orbit, Compression_Index, Sigma_classical_average, tt, t1, N
            )

            # Truncate each coefficient array to retain central 2*H+1 modes (±600)
            H = 600  # Keep a large number here; further truncation will occur during mode coupling
            center = int((len(Coefficient_tau_lambda_classical_Fourier1) - 1) / 2)
            Coefficient_tau_lambda_classical_Fourier1 = Coefficient_tau_lambda_classical_Fourier1[center - H : center + H + 1]
            Coefficient_tau_lambda_classical_Fourier2 = Coefficient_tau_lambda_classical_Fourier2[center - H : center + H + 1]

            # Clean up temporary variables
            del tt   
            # Note: 'f' is not defined in this scope — if unused, remove this line to avoid NameError
            # del f  # ← Uncomment only if 'f' exists upstream
            del t1 
            del N

            #==========================================================
            # Reconstruct higher-order terms via repeated FFT-based convolution
            # (equivalent to computing powers in the time domain)
            #========================================================== 
            # Power 1: square the base coefficient
            Coefficient_tau_lambda_classical_Fourier1_ = fft_convolve(
                Coefficient_tau_lambda_classical_Fourier1, 
                Coefficient_tau_lambda_classical_Fourier1
            )
            # Then convolve (Compression_Index - 2) more times to reach order = Compression_Index
            for i in range(Compression_Index - 2):
                Coefficient_tau_lambda_classical_Fourier1_ = fft_convolve(
                    Coefficient_tau_lambda_classical_Fourier1_, 
                    Coefficient_tau_lambda_classical_Fourier1
                )

            # Repeat for the second set of coefficients
            Coefficient_tau_lambda_classical_Fourier2_ = fft_convolve(
                Coefficient_tau_lambda_classical_Fourier2, 
                Coefficient_tau_lambda_classical_Fourier2
            )
            for i in range(Compression_Index - 2):
                Coefficient_tau_lambda_classical_Fourier2_ = fft_convolve(
                    Coefficient_tau_lambda_classical_Fourier2_, 
                    Coefficient_tau_lambda_classical_Fourier2
                )

            # Truncate reconstructed high-order coefficients to match the width of the original first-order term
            H = int((len(Coefficient_tau_lambda_classical_Fourier2) - 1) / 2)
            center = int((len(Coefficient_tau_lambda_classical_Fourier2_) - 1) / 2)
            Coefficient_tau_lambda_classical_Fourier1_ = Coefficient_tau_lambda_classical_Fourier1_[center - H : center + H + 1]
            Coefficient_tau_lambda_classical_Fourier2_ = Coefficient_tau_lambda_classical_Fourier2_[center - H : center + H + 1]

            #==========================================================
            # Group the two base expansions into positive/negative m₁ mode families
            #========================================================== 
            num_m_1 = len(Coeff_Phi22_mino_classical_Four)  # Number of m₁ modes from tidal tensor

            # Initialize power tables for m₁ ≥ 0 (positive frequencies)
            Coefficient_tau_lambda_classical_Fourier1_power = [[] for _ in range(int((num_m_1 - 1) / 2) + 1)]
            Coefficient_tau_lambda_classical_Fourier1_power[0] = [1]  # Identity element for convolution
            Coefficient_tau_lambda_classical_Fourier1_power[1] = Coefficient_tau_lambda_classical_Fourier1_

            Coefficient_tau_lambda_classical_Fourier2_power = [[] for _ in range(int((num_m_1 - 1) / 2) + 1)]
            Coefficient_tau_lambda_classical_Fourier2_power[0] = [1]
            Coefficient_tau_lambda_classical_Fourier2_power[1] = Coefficient_tau_lambda_classical_Fourier2_

            #==========================================================
            # Recursively compute higher powers via convolution
            #==========================================================
            for i in range(int((num_m_1 - 1) / 2) + 1):
                if i > 1:
                    Coefficient_tau_lambda_classical_Fourier1_power[i] = fft_convolve(
                        Coefficient_tau_lambda_classical_Fourier1_power[i - 1], 
                        Coefficient_tau_lambda_classical_Fourier1_
                    )
                    Coefficient_tau_lambda_classical_Fourier2_power[i] = fft_convolve(
                        Coefficient_tau_lambda_classical_Fourier2_power[i - 1], 
                        Coefficient_tau_lambda_classical_Fourier2_
                    )

            #==========================================================
            # Truncate all higher powers to the same spectral width as the first-order term
            #==========================================================
            for i in range(int((num_m_1 - 1) / 2) + 1):
                if i > 1:
                    H = int((len(Coefficient_tau_lambda_classical_Fourier1_power[1]) - 1) / 2)
                    center = int((len(Coefficient_tau_lambda_classical_Fourier1_power[i]) - 1) / 2)
                    Coefficient_tau_lambda_classical_Fourier1_power[i] = \
                        Coefficient_tau_lambda_classical_Fourier1_power[i][center - H : center + H + 1]
                    Coefficient_tau_lambda_classical_Fourier2_power[i] = \
                        Coefficient_tau_lambda_classical_Fourier2_power[i][center - H : center + H + 1]

            #==========================================================
            # Assemble a unified dictionary indexed by integer m₁ ∈ [−M, ..., M]
            # Positive m₁ use Coeff1, negative m₁ use Coeff2 (conjugate symmetry assumed)
            #==========================================================
            Coefficient_tau_lambda_classical_Fourier_m_1_power = {}
            num_m_1_d2 = int((num_m_1 - 1) / 2)

            for m_1 in range(-num_m_1_d2, num_m_1_d2 + 1):
                if m_1 >= 0:
                    Coefficient_tau_lambda_classical_Fourier_m_1_power[m_1] = Coefficient_tau_lambda_classical_Fourier1_power[m_1]
                else:
                    Coefficient_tau_lambda_classical_Fourier_m_1_power[m_1] = Coefficient_tau_lambda_classical_Fourier2_power[-m_1]

            # Explicitly define the zero-mode (m₁ = 0) as a delta function at zero frequency
            Coefficient_tau_lambda_classical_Fourier_m_1_power[0] = np.zeros(
                len(Coefficient_tau_lambda_classical_Fourier_m_1_power[1]), dtype=complex
            )
            center_idx = int((len(Coefficient_tau_lambda_classical_Fourier_m_1_power[1]) - 1) / 2)
            Coefficient_tau_lambda_classical_Fourier_m_1_power[0][center_idx] = 1.0 + 0.0j

        # Stage 11: Combine results to obtain the full Fourier decomposition in Mino time (τ)
        #===============================================
        for stage11 in range(1):
            #==========================================================
            # Reconstruct the total Fourier series for the Φ₂₂ and Φ₂₀ tidal components
            # by coupling orbital harmonics (m₁) with the time-reparameterization expansion.
            #========================================================== 
            num_m_1 = len(Coeff_Phi22_mino_classical_Four)
            num_m_1_d2 = int((num_m_1 - 1) / 2)
            
            # Determine the size of the zero-mode coefficient array (from τ-expansion)
            M_index = int(len(Coefficient_tau_lambda_classical_Fourier_m_1_power[0]))
            # Maximum orbital harmonic index
            N_max_classical = int((len(Coeff_Phi22_mino_classical_Four) - 1) / 2)

            # Initialize a zero-padded array to accumulate all shifted mode contributions
            Coefficient_table_classical = np.zeros(M_index + 2 * N_max_classical, dtype=complex)

            # Loop over all orbital harmonics m₁ ∈ [−N_max, ..., N_max]
            for m_1 in range(-N_max_classical, N_max_classical + 1):
                # Multiply the τ-expansion coefficient for mode m₁ by the corresponding tidal amplitude
                A = Coefficient_tau_lambda_classical_Fourier_m_1_power[m_1]
                A = A * Coeff_Phi22_mino_classical_Four[m_1 + N_max_classical]  # align index

                # Shift the spectrum by m₁ units in frequency space: 
                A = np.pad(
                    A, 
                    pad_width=(N_max_classical + m_1, N_max_classical - m_1), 
                    mode='constant', 
                    constant_values=0
                )
                Coefficient_table_classical += A

            # Repeat the same procedure for the axisymmetric Φ₂₀ component
            Coefficient_table_classical_1 = np.zeros(M_index + 2 * N_max_classical, dtype=complex)
            for m_1 in range(-N_max_classical, N_max_classical + 1):
                A = Coefficient_tau_lambda_classical_Fourier_m_1_power[m_1]
                A = A * Coeff_Phi20_mino_classical_Four[m_1 + N_max_classical]
                A = np.pad(
                    A, 
                    pad_width=(N_max_classical + m_1, N_max_classical - m_1), 
                    mode='constant', 
                    constant_values=0
                )
                Coefficient_table_classical_1 += A

            del A

            #==========================================================
            # Finalize frequency and coefficient arrays for downstream use
            #========================================================== 
            inner_num = int(len(Coefficient_table_classical))
            
            # Fundamental frequency in Mino time: ω₁ = 1 / ⟨Σ⟩
            omega1 = 1 / Sigma_classical_average

            # Construct the corresponding frequency grid: ωₙ = n ⋅ ω₁, where n ∈ ℤ
            pp = []
            for i in range(-int((inner_num - 1) / 2), int((inner_num - 1) / 2) + 1):
                pp.append(i)
            pp = np.array(pp)
            fre_table = pp * omega1
            fre_table = fre_table.flatten()

            # Assign the reconstructed Fourier coefficients to standard output variables
            Coefficient_Force_scalar_classical_table_1 = Coefficient_table_classical.flatten()
            Coefficient_Force_scalar_classical_table_2 = Coefficient_table_classical_1.flatten()
            

        # Stage 12: Convert dimensionless Fourier amplitudes into physical energy and angular momentum fluxes  
        #===============================================
        for stage12 in range(1):
            #==========================================================
            # Insert white dwarf physical parameters; k_rt has already been precomputed.
            # Transform the Mino-time frequencies to physical (stellar) frequencies.
            #==========================================================  
            # fre_table is in units of 1/⟨Σ⟩ (Mino time). To convert to physical frequency (rad/s),
            # we apply: ω_physical = ω_Mino * (R_BH / R_star) * sqrt(v_star / c)
            # where:
            #   ratio_sqrt_v_star_c = sqrt(v_star / c)  (dimensionless orbital velocity scale)
            #   ratio_R_star_R_BH   = R_star / R_BH     (size ratio)
            fre_table_star_classical = fre_table / ratio_sqrt_v_star_c * ratio_R_star_R_BH

            # The missing overall prefactor includes stellar response and geometric factors.
            # The core scaling for the tidal coupling is: (ratio_sqrt_v_star_c⁻² * k_rt³)²
            # where k_rt is the relativistic tidal Love number (or related response coefficient).

            #==========================================================
            # Compute angular momentum flux (dJ_z/dt) — proportional to m * ω⁵ * |F|²
            #==========================================================
            dotJ_z_dim_classical = 0
            mm = 2  # azimuthal number for the dominant ℓ=2, m=2 mode
            dotJ_z_dim_classical += mm * (
                (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
                * W22_square
                * np.abs(fre_table_star_classical)**5
                * np.abs(Coefficient_Force_scalar_classical_table_1)**2
            )

            #==========================================================
            # Compute orbital energy flux (dE_orb/dt) — proportional to (1/2) ω * dJ_z/dt per mode,
            # but summed over both m=2 (quadrupolar rotating) and m=0 (axisymmetric) components.
            #==========================================================
            dotE_orb_dim_classical = 0

            # Contribution from the m=2 (Φ₂₂) component
            dotE_orb_dim_classical += (0.5 * np.abs(fre_table_star_classical)) * mm * (
                (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
                * W22_square
                * np.abs(fre_table_star_classical)**5
                * np.abs(Coefficient_Force_scalar_classical_table_1)**2
            )

            # Contribution from the m=0 (Φ₂₀) component
            dotE_orb_dim_classical += (0.5 * np.abs(fre_table_star_classical)) * mm * (
                (ratio_sqrt_v_star_c**-2 * k_rt**3)**2
                * W20_square
                * np.abs(fre_table_star_classical)**5
                * np.abs(Coefficient_Force_scalar_classical_table_2)**2
            )

            #==========================================================
            # Rescale from stellar (white dwarf) frame back to the black hole background geometry
            # to obtain fluxes as measured in Boyer-Lindquist coordinate time.
            #==========================================================
            # Angular momentum flux rescaling: 
            dotJ_z = ratio_sqrt_v_star_c**2 * k_rt**-6 * dotJ_z_dim_classical
            dotJ_z_sum_classical = np.sum(dotJ_z)

            # Energy flux rescaling includes mass ratio (MBH/MWD) due to definition of reduced mass or normalization: 
            dotE_orb = ratio_sqrt_v_star_c**5 * k_rt**-6 * (MBH / MWD) * dotE_orb_dim_classical
            dotE_orb_sum_classical = np.sum(dotE_orb)

            # Note: The resulting dotJ_z_sum_classical and dotE_orb_sum_classical represent total fluxes
            #       evaluated in the black hole's coordinate time (Boyer-Lindquist time), not proper time.

        return ( (  rp_orbit, e_orbit, p_orbit,a_orbit,x_orbit, 
                    MBH, MWD,
                    dotJ_z_sum,    dotJ_z_sum_classical,
                    dotE_orb_sum,  dotE_orb_sum_classical,
                    E_orbit, L_orbit, Q_orbit  ) )  

    except Exception as e:  # Not a stable orbit 
        return ((  rp_orbit, e_orbit, p_orbit,a_orbit,x_orbit, MBH, MWD  )) 

In [10]:
# show result
def explain_results(result_tuple):
    """
    Convert the raw output of the orbital evolution function into a natural-language summary.
    
    Parameters:
    ----------
    result_tuple : tuple
        Expected order:
        (rp_orbit, e_orbit, p_orbit, a_orbit, x_orbit,
         MBH, MWD,
         dotJ_z_sum, dotJ_z_sum_classical,
         dotE_orb_sum, dotE_orb_sum_classical,
         E_orbit, L_orbit, Q_orbit)
    """
    (
        rp_orbit, e_orbit, p_orbit, a_orbit, x_orbit,
        MBH, MWD,
        dotJ_z_sum, dotJ_z_sum_classical,
        dotE_orb_sum, dotE_orb_sum_classical,
        E_orbit, L_orbit, Q_orbit
    ) = result_tuple

    print("=" * 53)
    print("Orbital Evolution Summary")
    print("=" * 53)

    # --- Orbital parameters ---
    print(f"• Pericenter distance:        r_p = {rp_orbit:.3f} r_g")
    print(f"• Eccentricity:                 e = {e_orbit:.4f}")
    print(f"• Semi-latus rectum:            p = {p_orbit:.3f} r_g")
    
    print(f"• Black hole spin parameter:    a = {a_orbit:.3f} M_BH")
    print(f"• Orbital inclination parameter:x = {x_orbit:.5f}")

    # --- Conserved quantities ---
    print(f"\n• Specific orbital energy:      E = {E_orbit:.5f} c²")
    print(f"• Specific angular momentum:    L = {L_orbit:.3f} r_g c")
    print(f"• Carter constant:              Q = {Q_orbit:.3f} (r_g c)²")
    
    # --- Masses ---
    print(f"\n• Black hole mass:           M_BH = {MBH:.2e} M_⊙")
    print(f"• White dwarf mass:          M_WD = {MWD:.2e} M_⊙")

    # --- Angular momentum loss ---
    print("\nSpecific angular momentum loss rate:")
    print("• Total dJ_z/dt (GR tidal):        {:.3e}".format(dotJ_z_sum))
    print("• Classical dJ_z/dt (Newtonian):   {:.3e}".format(dotJ_z_sum_classical)) 

    # --- Orbital energy loss ---
    print("\nSpecific energy loss rate:")
    print("• Total dE_orb/dt (GR tidal):      {:.3e}".format(dotE_orb_sum))
    print("• Classical dE_orb/dt (Newtonian): {:.3e}".format(dotE_orb_sum_classical)) 

    print("=" * 53)

In [11]:
rp_val=12.0
e_val=0.5
p_val=rp_val*(1+e_val) 

result = tidal_heating_function(
            (   rp_val ,
                e_val,
                p_val,
                0.5, # a_orbit
                1,   # x_orbit 
                # Since the results are only applicable to equatorial orbit, please refrain from modifying the value of x_orbit = 1 
                1e5, # MBH 
                0.6, # MWD
                ratio_sqrt_v_star_c, # sqrt(GM/c^2/R)
                ratio_R_star_R_BH  # ratio_R_star_R_BH
                ) 
        )  
explain_results(result)    

Orbital Evolution Summary
• Pericenter distance:        r_p = 12.000 r_g
• Eccentricity:                 e = 0.5000
• Semi-latus rectum:            p = 18.000 r_g
• Black hole spin parameter:    a = 0.500 M_BH
• Orbital inclination parameter:x = 1.00000

• Specific orbital energy:      E = 0.97977 c²
• Specific angular momentum:    L = 4.584 r_g c
• Carter constant:              Q = 0.000 (r_g c)²

• Black hole mass:           M_BH = 1.00e+05 M_⊙
• White dwarf mass:          M_WD = 6.00e-01 M_⊙

Specific angular momentum loss rate:
• Total dJ_z/dt (GR tidal):        8.155e-06
• Classical dJ_z/dt (Newtonian):   8.926e-06

Specific energy loss rate:
• Total dE_orb/dt (GR tidal):      3.429e-07
• Classical dE_orb/dt (Newtonian): 4.033e-07


Due to different computers handling parallel computation differently, the process of parallel computation for 1D list  " flattened_rp_e_list "  will not be demonstrated here